In [ ]:
from grading.grader import grade_answer

In [ ]:
from grading.grader import grade_answer
import re
import os
import json


flags = [
    "Analysis.Problem_Definition",
    "Analysis.Information_Organization",
    "Analysis.Problem_Structuring",
    "Inference.Deductive_Reasoning",
    "Inference.Inductive_Reasoning",
    "Inference.Abductive_Reasoning",
    "Judgment.Principle_Selection",
    "Judgment.Evaluation_of_Alternatives",
    "Judgment.Conclusion_Decision",
    "Suggestion.Strategic_Planning",
    "Suggestion.Branch_Changing",
    "Suggestion.Hypothesis_Generation",
    "Suggestion.Analogy_Recall",
    "Reflection.Self_Monitoring_Evaluation",
    "Reflection.Counterfactual_Thinking",
    "Reflection.Causal_Attribution",
    "Reflection.Strategy_Regulation"
]

# flag
flag_abbreviations = {
    "Analysis.Problem_Definition": "A.PD",
    "Analysis.Information_Organization": "A.IO", 
    "Analysis.Problem_Structuring": "A.PS",
    "Inference.Deductive_Reasoning": "I.DR",
    "Inference.Inductive_Reasoning": "I.IR",
    "Inference.Abductive_Reasoning": "I.AR",
    "Judgment.Principle_Selection": "J.PS",
    "Judgment.Evaluation_of_Alternatives": "J.EA",
    "Judgment.Conclusion_Decision": "J.CD",
    "Suggestion.Strategic_Planning": "S.SP",
    "Suggestion.Branch_Changing": "S.BC",
    "Suggestion.Hypothesis_Generation": "S.HG",
    "Suggestion.Analogy_Recall": "S.AR",
    "Reflection.Self_Monitoring_Evaluation": "R.SME",
    "Reflection.Counterfactual_Thinking": "R.CT",
    "Reflection.Causal_Attribution": "R.CA",
    "Reflection.Strategy_Regulation": "R.SR"
}

# Create reverse mapping from abbreviations to full names
abbreviation_to_flag = {v: k for k, v in flag_abbreviations.items()}


# def latex_equal(latex1, latex2):
#     try:
#         if latex1 is None or latex2 is None: # Check if either input is None
#             return False
#         expr1 = parse_latex(latex1)
#         expr2 = parse_latex(latex2)
#         return expr1.equals(expr2)
#     except Exception as e:
#         print(f"Error parsing LaTeX expressions: {e}")
#         return False
def latex_equal(latex1, latex2):
    try:
        return grade_answer(latex1, latex2)
    except Exception as e:
        print(f"Error comparing LaTeX expressions: {e}")
        return False
    
def extract_answer(response):
    '''
    Extracts the answer by finding the last \boxed equation
    '''
    # Ensure response is a string before using regex
    if not isinstance(response, str):
        return None
    
    # Find all instances of \boxed{...} using regex
    # We need to handle nested braces properly
    boxed_pattern = r'\\boxed\{'
    
    # Find all starting positions of \boxed{
    boxed_starts = []
    for match in re.finditer(boxed_pattern, response):
        boxed_starts.append(match.start())
    
    if not boxed_starts:
        return [response]
    
    # For each \boxed{, find the matching closing brace
    boxed_contents = []
    
    for start_pos in boxed_starts:
        # Find the opening brace position
        brace_start = response.find('{', start_pos)
        if brace_start == -1:
            continue
            
        # Count braces to find the matching closing brace
        brace_count = 0
        pos = brace_start
        
        while pos < len(response):
            if response[pos] == '{':
                brace_count += 1
            elif response[pos] == '}':
                brace_count -= 1
                if brace_count == 0:
                    # Found the matching closing brace
                    content = response[brace_start + 1:pos]
                    boxed_contents.append(content)
                    break
            pos += 1
    
    # Return the last boxed content if any were found
    if boxed_contents:
        return boxed_contents
    else:
        return None
    
def get_all_records(path, use_human_anot=False, filter=None):
    """
    Retrieves all records, and can choose to use human labels or machine labels based on the use_human_anot parameter.
    
    Parameters:
    path -- Data file path
    use_human_anot -- Whether to use human labels. If True, uses the human_anot field as labels;
                     If False, uses the used_flag field as labels.
    """
    files = []
    answers = {}
    for root, dirs, filenames in os.walk(path):
        
        for filename in filenames:
            # if 'Peter' in filename:
            #     continue
            if filter is not None and filter(os.path.join(root,filename)):
                continue
            
            answer = {}
            if filename.endswith('.json') and 'embeddings' not in filename:
                files.append(os.path.join(root, filename))
                with open(os.path.join(root, filename), 'r') as f:
                    data = f.read()
                    data = json.loads(data)
                answer['steps'] = {}
                answer['question'] = data['question']
                #answer['answer'] = data['answer']
                answer['ground_truth'] = data['True_Answer']
                #answer['reasoning'] = data['reasoning']
                if 'reasoning' not in data.keys():
                    # concat ['steps']['reasoning'][i]['content'] for reasoning
                    reasoning = []
                    for step, content in data['steps']['reasoning'].items():
                        try:
                            _id = int(step)
                        except:
                            continue
                        reasoning.append(content['content'])
                    answer['reasoning'] = ' '.join(reasoning)
                else:
                    answer['reasoning'] = data['reasoning']
                for step, content in data['steps']['reasoning'].items():
                    try:
                        _id = int(step)
                    except:
                        continue
                    #print(content)
                    used_flag = 'human_anot' if isinstance(use_human_anot,bool) and use_human_anot else use_human_anot + '_flag'
                    answer['steps'][_id] = {
                        'content': content['content'],
                        'used_flag': content.get(used_flag, None),
                        #'human_anot': content.get('human_anot', None),
                        # Save the selected label
                        'selected_flag': used_flag 
                    }     
                answers[os.path.join(root, filename)] = answer
                
    return answers





In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties ,fontManager
import seaborn as sns
import numpy as np
import pandas as pd

# Times New Roman，
font_path = './fonts/times.ttf'
fontManager.addfont(path=font_path)
prop = FontProperties(fname=font_path)

sns.set(context='paper', 
        style='ticks', 
        palette='deep', 
        font=prop.get_name(),
        font_scale=2.8, 
        rc={
            'mathtext.fontset': 'stix',
            'pdf.fonttype': 42,
            'lines.linewidth' : 4,
            'lines.markersize' : 8,
            'font.weight': 'bold',
            'axes.labelweight': 'bold',
            'axes.titleweight': 'bold',
            'figure.titleweight': 'bold'
        }
)
sns.despine()

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
import os

def partition_samples(data):
    correct_records = {}
    incorrect_records = {}
    manual_check = [('Math','5981',True),('Math','1582',True),('Math','1910',True),('Math','3547',True),('Math','4648',True),
                    ('Math','4613',True),('Math','2198',True),('Math','2082',True),('Math','4456',True),('Math','6287',True),
                    ('Math','6270',True),('Math','5091',True)]
    
    
    for file, record in data.items():
        flag = False
        for check in manual_check:
            if check[0].lower() in file.lower() and check[1]+'.json' in file:
                if check[2]:
                    correct_records[file] = record
                else:
                    incorrect_records[file] = record
                flag = True
                break
        if flag:
            continue
        
        try:
            answer = record['ground_truth']
            reasoning = record['reasoning']
            # try to match choices
            # if '\\text' in answer:
            #     _answer = _answer.replace('\\text', '\\boxed')
            _answer = extract_answer(answer)[-1]
        except Exception as e:
            #print(f"Error extracting answer from {file}: {e}")
            continue
        try:
            if 'Final Answer' in reasoning:
                _reasoning = ','.join(extract_answer(reasoning.split('Final Answer')[-1]))
            else:
                _reasoning = extract_answer(reasoning)[-1]
        except Exception as e:
            #print(f"Error extracting reasoning from {file}: {e}")
            continue
        if_correct = latex_equal(_answer, _reasoning)
        # fix for choices, if not if_correct and _answer exist in _reasoning[-100], also correct:
        if not if_correct and _answer is not None:
            if _reasoning in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
                # match _answer to choice in question
                beginning = record['question'].find(_answer)
                choice = re.findall(r'\b[A-Z]\b', record['question'][:beginning])
                if len(choice) > 0:
                    # if the last choice matches _reasoning, then it is correct
                    if_correct = choice[-1] == _reasoning
            else:
                pass
        
        if if_correct:
            correct_records[file] = record
        else:
            incorrect_records[file] = record
    print(f"Total records: {len(data)}")
    print(f"Correct records: {len(correct_records)}")
    print(f"Incorrect records: {len(incorrect_records)}")
    
    # calculate total steps in correct and incorrect records
    total_correct_steps = sum(len(record['steps']) for record in correct_records.values())
    total_incorrect_steps = sum(len(record['steps']) for record in incorrect_records.values())
    print(f"Total steps in correct records: {total_correct_steps}")
    print(f"Total steps in incorrect records: {total_incorrect_steps}")
    return correct_records, incorrect_records


def analysis_once(data):
    correct_records, incorrect_records = partition_samples(data)
    
    # Analyze the distribution of flags in each record

    def count_flags_in_record(record):
        """Count the occurrences of each flag in a record"""
        flag_counts = {flag: 0 for flag in flags}
        total_steps = 0
        
        for step_id, step_content in record['steps'].items():
            selected_flags = step_content.get('used_flag')
            if selected_flags:
                # Handle flags that might be in string format
                if isinstance(selected_flags, str):
                    try:
                        selected_flags = eval(selected_flags)
                    except:
                        selected_flags = [selected_flags]
                elif not isinstance(selected_flags, list):
                    selected_flags = [selected_flags]
                    
                for flag in selected_flags:
                    if flag in flag_counts:
                        flag_counts[flag] += 1
                total_steps += 1
        
        # Return the count of each flag and total number of steps
        return flag_counts, total_steps

    # Count the total occurrences of each flag in correct and incorrect records
    correct_flag_counts = {flag: 0 for flag in flags}
    correct_total_steps = 0
    incorrect_flag_counts = {flag: 0 for flag in flags}
    incorrect_total_steps = 0

    correct_files = {}
    incorrect_files = {}

    for file, record in correct_records.items():
        flag_counts, total_steps = count_flags_in_record(record)
        correct_files[file] = {k: v/ total_steps for k, v in flag_counts.items()}
        for flag, count in flag_counts.items():
            correct_flag_counts[flag] += count
        correct_total_steps += total_steps
        
    for file, record in incorrect_records.items():
        flag_counts, total_steps = count_flags_in_record(record)
        incorrect_files[file] = {k: v / total_steps for k, v in flag_counts.items()}
        for flag, count in flag_counts.items():
            incorrect_flag_counts[flag] += count
        incorrect_total_steps += total_steps
    

    # Convert file-level flag proportion data to DataFrame for analysis
    correct_df = pd.DataFrame.from_dict(correct_files, orient='index')
    incorrect_df = pd.DataFrame.from_dict(incorrect_files, orient='index')

    # Data structure for storing test results
    test_results = []

    # Hypothesis testing for each flag
    for flag in flags:
        correct_values = correct_df[flag].dropna().values
        incorrect_values = incorrect_df[flag].dropna().values
        
        # Skip if there's too little data in either group
        if len(correct_values) < 5 or len(incorrect_values) < 5:
            print(f"Skipping {flag} due to insufficient sample size")
            continue
        
        # 1. F-test (variance test)
        f_stat = np.var(correct_values, ddof=1) / np.var(incorrect_values, ddof=1)
        df1 = len(correct_values) - 1
        df2 = len(incorrect_values) - 1
        
        if np.var(correct_values) > np.var(incorrect_values):
            p_value_var = 2 * (1 - stats.f.cdf(f_stat, df1, df2))
        else:
            p_value_var = 2 * stats.f.cdf(f_stat, df1, df2)
        
        var_equal = p_value_var > 0.01  # Whether variances are equal
        
        # 2. Mann-Whitney U test (non-parametric test)
        u_stat, p_value_u = stats.mannwhitneyu(
            correct_values,
            incorrect_values,
            alternative='two-sided'
        )
        
        # 3. Calculate confidence intervals using bootstrap method
        # Bootstrap confidence interval for the difference in means
        n_bootstrap = 1000
        np.random.seed(42)  # For reproducibility
        
        bootstrap_diffs = []
        for _ in range(n_bootstrap):
            # Bootstrap sample from each group independently
            correct_bootstrap = np.random.choice(correct_values, size=len(correct_values), replace=True)
            incorrect_bootstrap = np.random.choice(incorrect_values, size=len(incorrect_values), replace=True)
            bootstrap_diff = np.mean(correct_bootstrap) - np.mean(incorrect_bootstrap)
            bootstrap_diffs.append(bootstrap_diff)
        
        # Calculate 99% confidence interval
        ci_low = np.percentile(bootstrap_diffs, 0.5)
        ci_high = np.percentile(bootstrap_diffs, 99.5)
        
        # Calculate effect size: Cohen's d (still useful for effect size interpretation)
        mean_diff = np.mean(correct_values) - np.mean(incorrect_values)
        pooled_std = np.sqrt(((len(correct_values) - 1) * np.var(correct_values, ddof=1) + 
                            (len(incorrect_values) - 1) * np.var(incorrect_values, ddof=1)) / 
                            (len(correct_values) + len(incorrect_values) - 2))
        cohen_d = mean_diff / pooled_std
        
        # Store results
        test_results.append({
            'Flag': flag,
            'Mean_Correct': np.mean(correct_values),
            'Mean_Incorrect': np.mean(incorrect_values),
            'Mean_Diff': mean_diff,
            'Std_Correct': np.std(correct_values, ddof=1),
            'Std_Incorrect': np.std(incorrect_values, ddof=1),
            'F_Stat': f_stat,
            'P_Value_Var': p_value_var,
            'Var_Equal': var_equal,
            'U_Stat': u_stat,
            'P_Value_U': p_value_u,
            'CI_Low': ci_low,
            'CI_High': ci_high,
            'Cohen_d': cohen_d,
            'Significant_U': p_value_u < 0.01
        })

    # Convert results to DataFrame
    results_df = pd.DataFrame(test_results)
    
    # figures
    os.makedirs('figures', exist_ok=True)

    # Visualize mean differences and confidence intervals
    fig, ax1 = plt.subplots(figsize=(14, 8))

    # Calculate error bar upper and lower limits (relative to mean difference)
    yerr = np.zeros((2, len(results_df)))
    yerr[0, :] = results_df['Mean_Diff'] - results_df['CI_Low']
    yerr[1, :] = results_df['CI_High'] - results_df['Mean_Diff']
    
    # Ensure yerr values are non-negative (fix for matplotlib requirement)
    yerr[0, :] = np.abs(yerr[0, :])
    yerr[1, :] = np.abs(yerr[1, :])
    
    # Check for any problematic values
    if np.any(yerr < 0):
        print("Warning: Found negative yerr values, taking absolute values")
        yerr = np.abs(yerr)

    # Multiple comparison correction (because we performed multiple hypothesis tests)
    # Using Benjamini-Hochberg method to correct p-values (False Discovery Rate FDR) for U-test
    _, corrected_p_values_u, _, _ = multipletests(results_df['P_Value_U'].values, alpha=0.01, method='fdr_bh')
    results_df['P_Value_U_Corrected'] = corrected_p_values_u
    results_df['U_Significant_Corrected'] = corrected_p_values_u < 0.01

    # 
    flag_abbr_labels = [flag_abbreviations[flag] for flag in results_df['Flag']]

    # Plot mean differences and confidence intervals with significance highlighting
    # Create color array based on significance
    point_colors = ['red' if sig else 'blue' for sig in results_df['U_Significant_Corrected']]
    
    # Plot each point individually to control color
    for i in range(len(results_df)):
        ax1.errorbar(
            i,
            results_df['Mean_Diff'].iloc[i],
            yerr=[[yerr[0, i]], [yerr[1, i]]],
            fmt='o',
            capsize=5,
            color=point_colors[i],
            ecolor='black',
            markersize=8
        )
    
    ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.7)
    ax1.set_xticks(range(len(results_df)))
    ax1.set_xticklabels(flag_abbr_labels, rotation=45)
    ax1.set_ylabel('Mean Difference (Correct - Incorrect)')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    
    #  (results_dfflag)
    # results_dfflag
    relative_abundance = []
    for flag in results_df['Flag']:
        correct_mean = correct_df[flag].values
        incorrect_mean = incorrect_df[flag].values
        # get mean of correct and incorrect values
        relative_abundance.append((np.sum(correct_mean) + np.sum(incorrect_mean)) / (len(correct_mean) + len(incorrect_mean)))
    
    # Y
    ax2 = ax1.twinx()
    ax2.plot(range(len(results_df)), relative_abundance, 
             color='purple', linewidth=3, marker='o', markersize=6, 
             label=None, alpha=0.8)
    ax2.set_ylabel('Proportion', color='purple')
    ax2.tick_params(axis='y', labelcolor='purple')
    
    # Add legend for significance
    import matplotlib.patches as mpatches
    red_patch = mpatches.Patch(color='red', label='Significant (p < 0.01)')
    blue_patch = mpatches.Patch(color='blue', label='Not Significant')
    purple_line = plt.Line2D([0], [0], color='purple', linewidth=3, label='Relative Abundance')
    ax1.legend(handles=[red_patch, blue_patch], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('figures/mean_differences_confidence_intervals.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # Visualize distribution of each flag in correct and incorrect samples
    plt.figure(figsize=(20, 10))

    bar_width = 0.35
    index = np.arange(len(flags))

    # Calculate means and standard deviations for both groups
    correct_means = [np.mean(correct_df[flag].values) if flag in correct_df else 0 for flag in flags]
    incorrect_means = [np.mean(incorrect_df[flag].values) if flag in incorrect_df else 0 for flag in flags]
    correct_stds = [np.std(correct_df[flag].values, ddof=1) if flag in correct_df else 0 for flag in flags]
    incorrect_stds = [np.std(incorrect_df[flag].values, ddof=1) if flag in incorrect_df else 0 for flag in flags]

    # 
    flag_short_names = [flag_abbreviations[f] for f in flags]

    # Draw bar chart
    plt.bar(index, correct_means, bar_width, label='Correct', color='green', alpha=0.7)
    plt.bar(index + bar_width, incorrect_means, bar_width, label='Incorrect', color='red', alpha=0.7)

    # Add error bars
    plt.errorbar(index, correct_means, yerr=correct_stds, fmt='none', ecolor='black', capsize=3)
    plt.errorbar(index + bar_width, incorrect_means, yerr=incorrect_stds, fmt='none', ecolor='black', capsize=3)

    plt.xlabel('Flag Categories')
    plt.ylabel('Average Proportion')
    plt.xticks(index + bar_width / 2, flag_short_names, rotation=90)
    plt.legend()
    plt.tight_layout()
    plt.savefig('figures/flag_distribution_comparison.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # Create a heatmap showing correlation between flags in both groups
    plt.figure(figsize=(16, 14))

    # Merge DataFrames and calculate correlations
    combined_df = pd.concat([correct_df.assign(group='correct'), 
                            incorrect_df.assign(group='incorrect')])
    correlation_matrix = combined_df.drop(columns=['group']).corr()
    
    # 
    correlation_matrix.columns = [flag_abbreviations[col] for col in correlation_matrix.columns]
    correlation_matrix.index = [flag_abbreviations[idx] for idx in correlation_matrix.index]

    # Draw heatmap
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, center=0.0)
    plt.tight_layout()
    plt.savefig('figures/flag_correlation_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Multiple comparison correction (because we performed multiple hypothesis tests)

    # Using Benjamini-Hochberg method to correct p-values (False Discovery Rate FDR) for U-test
    _, corrected_p_values_u, _, _ = multipletests(results_df['P_Value_U'].values, alpha=0.01, method='fdr_bh')
    results_df['P_Value_U_Corrected'] = corrected_p_values_u
    results_df['U_Significant_Corrected'] = corrected_p_values_u < 0.01

    # Display results after p-value correction
    print("\nSignificance results after p-value correction:")
    # 
    display_df = results_df.copy()
    display_df['Flag_Abbr'] = [flag_abbreviations[flag] for flag in display_df['Flag']]
    print(display_df[['Flag_Abbr', 'Mean_Diff', 'P_Value_U', 'P_Value_U_Corrected', 'U_Significant_Corrected']])

    # Visualize flags by category
    categories = {}
    for flag in flags:
        category = flag.split('.')[0]
        subcategory = flag.split('.')[1]
        if category not in categories:
            categories[category] = []
        categories[category].append(flag)

    # Create a chart for each category
    for category, category_flags in categories.items():
        # Filter results for the current category
        category_results = results_df[results_df['Flag'].isin(category_flags)]
        
        if len(category_results) == 0:
            print(f"Category {category} does not have enough data for visualization")
            continue
        
        # 
        subcategories = [flag_abbreviations[f] for f in category_results['Flag']]
        
        plt.figure(figsize=(18, 10))
        
        # Set up subplot grid
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Subplot 1: Mean comparison
        bar_width = 0.35
        index = np.arange(len(category_results))
        
        axes[0].bar(index, category_results['Mean_Correct'], bar_width, label='Correct', color='green', alpha=0.7)
        axes[0].bar(index + bar_width, category_results['Mean_Incorrect'], bar_width, label='Incorrect', color='red', alpha=0.7)
        
        axes[0].set_xlabel('Subcategories')
        axes[0].set_ylabel('Average Proportion')
        axes[0].set_xticks(index + bar_width / 2)
        axes[0].set_xticklabels(subcategories, rotation=45, ha='right')
        axes[0].legend()
        
        # Subplot 2: Confidence intervals
        yerr = np.zeros((2, len(category_results)))
        yerr[0, :] = category_results['Mean_Diff'] - category_results['CI_Low']
        yerr[1, :] = category_results['CI_High'] - category_results['Mean_Diff']
        
        # Ensure yerr values are non-negative (fix for matplotlib requirement)
        yerr[0, :] = np.abs(yerr[0, :])
        yerr[1, :] = np.abs(yerr[1, :])
        
        # Check for any problematic values
        if np.any(yerr < 0):
            print(f"Warning: Found negative yerr values in {category}, taking absolute values")
            yerr = np.abs(yerr)
        
        axes[1].errorbar(
            range(len(category_results)),
            category_results['Mean_Diff'],
            yerr=yerr,
            fmt='o',
            capsize=5,
            color='blue',
            ecolor='black',
            markersize=8
        )
        axes[1].axhline(y=0, color='red', linestyle='-', alpha=0.7)
        axes[1].set_xticks(range(len(category_results)))
        axes[1].set_xticklabels(subcategories, rotation=45, ha='right')
        axes[1].set_ylabel('Mean Difference (Correct - Incorrect)')
        axes[1].grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add statistical significance markers
        for i, (idx, row) in enumerate(category_results.iterrows()):
            if row['U_Significant_Corrected']:
                if row['Mean_Diff'] > 0:
                    y_pos = row['Mean_Diff'] + yerr[1, i] + 0.01
                else:
                    y_pos = row['Mean_Diff'] - yerr[0, i] - 0.03
                axes[1].text(i, y_pos, '*', ha='center', va='center', fontsize=16)
        
        plt.tight_layout()
        plt.savefig(f'figures/category_{category}_analysis.pdf', dpi=300, bbox_inches='tight')
        plt.show()

    # Visualize the importance of each flag for correct/incorrect classification
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    from sklearn.inspection import permutation_importance

    # Prepare data
    X = pd.concat([correct_df, incorrect_df])
    y = np.array([1] * len(correct_df) + [0] * len(incorrect_df))

    # Train a random forest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X, y)

    # Calculate feature importance
    importances = clf.feature_importances_
    indices = np.argsort(importances)[::-1]

    # 
    flag_abbr_for_importance = [flag_abbreviations[X.columns[i]] for i in indices]

    # Plot feature importance
    plt.figure(figsize=(16, 10))
    plt.bar(range(X.shape[1]), importances[indices], color='b', align='center')
    plt.xticks(range(X.shape[1]), flag_abbr_for_importance, rotation=90)
    plt.tight_layout()
    plt.savefig('figures/random_forest_importance.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # Calculate permutation importance
    result = permutation_importance(clf, X, y, n_repeats=10, random_state=42)
    sorted_idx = result.importances_mean.argsort()[::-1]

    # 
    flag_abbr_for_permutation = [flag_abbreviations[X.columns[i]] for i in sorted_idx]

    # Plot permutation importance
    plt.figure(figsize=(16, 10))
    plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=flag_abbr_for_permutation)
    plt.tight_layout()
    plt.savefig('figures/permutation_importance.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # Create comprehensive results table
    comprehensive_results = pd.DataFrame({
        'Flag': flags,
        'Flag_Abbr': [flag_abbreviations[flag] for flag in flags],
        'Mean_Correct': [np.mean(correct_df[flag].values) if flag in correct_df else np.nan for flag in flags],
        'Mean_Incorrect': [np.mean(incorrect_df[flag].values) if flag in incorrect_df else np.nan for flag in flags],
        'Std_Correct': [np.std(correct_df[flag].values, ddof=1) if flag in correct_df else np.nan for flag in flags],
        'Std_Incorrect': [np.std(incorrect_df[flag].values, ddof=1) if flag in incorrect_df else np.nan for flag in flags],
        'P_Value_U': [results_df[results_df['Flag'] == flag]['P_Value_U'].values[0] if flag in results_df['Flag'].values else np.nan for flag in flags],
        'P_Value_U_Corrected': [results_df[results_df['Flag'] == flag]['P_Value_U_Corrected'].values[0] if flag in results_df['Flag'].values else np.nan for flag in flags],
        'U_Significant': [results_df[results_df['Flag'] == flag]['U_Significant_Corrected'].values[0] if flag in results_df['Flag'].values else False for flag in flags],
        'Effect_Size': [results_df[results_df['Flag'] == flag]['Cohen_d'].values[0] if flag in results_df['Flag'].values else np.nan for flag in flags],
        'RF_Importance': [importances[list(X.columns).index(flag)] if flag in X.columns else np.nan for flag in flags],
    })

    # Sort results by significance and effect size
    comprehensive_results = comprehensive_results.sort_values(by=['U_Significant', 'Effect_Size'], ascending=[False, False])

    # Display final results table
    pd.set_option('display.max_rows', None)
    print("\nComprehensive Analysis Results:")
    # 
    print(comprehensive_results[['Flag_Abbr', 'Mean_Correct', 'Mean_Incorrect', 'P_Value_U_Corrected', 'U_Significant', 'Effect_Size', 'RF_Importance']])
    
    # Create a DataFrame with flags and their significance
    significance_df = results_df[['Flag', 'U_Significant_Corrected']]

    # Create a count of flags that are significant
    u_significant = significance_df[significance_df['U_Significant_Corrected']].shape[0]
    none_significant = significance_df[~significance_df['U_Significant_Corrected']].shape[0]

    # Create a bar chart comparing test results
    fig, ax = plt.subplots(figsize=(10, 7))

    categories = ['U-test Significant', 'Not Significant']
    counts = [u_significant, none_significant]
    colors = ['green', 'gray']

    bars = ax.bar(categories, counts, color=colors)

    # Add count labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=12)

    plt.ylabel('Number of Flags')
    plt.tight_layout()
    plt.savefig('figures/significance_summary.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # List all significant flags from U-test
    all_significant_flags = results_df[results_df['U_Significant_Corrected']]['Flag'].tolist()
    print(f"\nTotal number of flags found significant by U-test: {len(all_significant_flags)}")
    print("Flags significant in U-test:")
    for flag in all_significant_flags:
        flag_row = results_df[results_df['Flag'] == flag]
        u_sig = "✓" if flag_row['U_Significant_Corrected'].values[0] else "✗"
        print(f"- {flag_abbreviations[flag]}: U-test [{u_sig}]")

In [ ]:
# stats for normality

def normality_test(data):
    """
    Perform normality tests on the data.
    
    Parameters:
    data -- DataFrame containing the data to test for normality.
    
    Returns:
    results -- DataFrame with normality test results.
    """
    results = []
    _data = data.copy()
    # test the proportions of each flag is obey
    correct_records, incorrect_records = partition_samples(_data)
    
    # Analyze the distribution of flags in each record

    def count_flags_in_record(record):
        """Count the occurrences of each flag in a record"""
        flag_counts = {flag: 0 for flag in flags}
        total_steps = 0
        
        for step_id, step_content in record['steps'].items():
            selected_flags = step_content.get('used_flag')
            if selected_flags:
                # Handle flags that might be in string format
                if isinstance(selected_flags, str):
                    try:
                        selected_flags = eval(selected_flags)
                    except:
                        selected_flags = [selected_flags]
                elif not isinstance(selected_flags, list):
                    selected_flags = [selected_flags]
                    
                for flag in selected_flags:
                    if flag in flag_counts:
                        flag_counts[flag] += 1
                total_steps += 1
        
        # Return the count of each flag and total number of steps
        return flag_counts, total_steps

    # Count the total occurrences of each flag in correct and incorrect records
    correct_flag_counts = {flag: 0 for flag in flags}
    correct_total_steps = 0
    incorrect_flag_counts = {flag: 0 for flag in flags}
    incorrect_total_steps = 0

    correct_files = {}
    incorrect_files = {}

    for file, record in correct_records.items():
        flag_counts, total_steps = count_flags_in_record(record)
        correct_files[file] = {k: v/ total_steps for k, v in flag_counts.items()}
        for flag, count in flag_counts.items():
            correct_flag_counts[flag] += count
        correct_total_steps += total_steps
        
    for file, record in incorrect_records.items():
        flag_counts, total_steps = count_flags_in_record(record)
        incorrect_files[file] = {k: v / total_steps for k, v in flag_counts.items()}
        for flag, count in flag_counts.items():
            incorrect_flag_counts[flag] += count
        incorrect_total_steps += total_steps
    

    # Convert file-level flag proportion data to DataFrame for analysis
    correct_df = pd.DataFrame.from_dict(correct_files, orient='index')
    incorrect_df = pd.DataFrame.from_dict(incorrect_files, orient='index')

    # test if the distribution of each flag is normal in each group
    for flag in flags:
        correct_values = correct_df[flag].dropna().values
        incorrect_values = incorrect_df[flag].dropna().values
        
        if len(correct_values) < 5 or len(incorrect_values) < 5:
            print(f"Skipping {flag} due to insufficient sample size")
            continue
        
        # Perform Shapiro-Wilk test for normality
        shapiro_correct = stats.shapiro(correct_values)
        shapiro_incorrect = stats.shapiro(incorrect_values)
        
        # Perform Kolmogorov-Smirnov test for normality
        ks_correct = stats.kstest(correct_values, 'norm', args=(np.mean(correct_values), np.std(correct_values)))
        ks_incorrect = stats.kstest(incorrect_values, 'norm', args=(np.mean(incorrect_values), np.std(incorrect_values)))
        
        results.append({
            'Flag': flag,
            'Shapiro_Wilk_Stat_Correct': shapiro_correct.statistic,
            'Shapiro_Wilk_P_Correct': shapiro_correct.pvalue,
            'Shapiro_Wilk_Stat_Incorrect': shapiro_incorrect.statistic,
            'Shapiro_Wilk_P_Incorrect': shapiro_incorrect.pvalue,
            'KS_Stat_Correct': ks_correct.statistic,
            'KS_P_Correct': ks_correct.pvalue,
            'KS_Stat_Incorrect': ks_incorrect.statistic,
            'KS_P_Incorrect': ks_incorrect.pvalue
        })
    # visualize the results
    results_df = pd.DataFrame(results)
    return pd.DataFrame(results)

In [ ]:
data = get_all_records('./machine_annotations',use_human_anot='gemini-best')
normality_test_res = normality_test(data)
print("Normality Test Results:")
print(normality_test_res)
analysis_once(data)

In [ ]:
# Particular analyses
# 1. get the average position of S.AR and S.HG in correct and incorrect records, respectively
# Here, position refers to the normalized step number of S.AR and S.HG, if a S.AR or S.HG step is not present, then the position is -1.


def average_position_analysis(data, flag):
    correct, incorrect = partition_samples(data)
    correct_positions = []
    incorrect_positions = []
    for record in correct.values():
        steps = record['steps']
        _tmp = []
        for step_id, step_content in steps.items():
            if flag in step_content.get('used_flag', []):
                # Normalize the position by the total number of steps
                position = int(step_id) / len(steps)
                _tmp.append(position)
        if len(_tmp) > 0:
            avg_position = np.mean(_tmp)
            correct_positions.append(avg_position) 
    for record in incorrect.values():
        steps = record['steps']
        _tmp = []
        for step_id, step_content in steps.items():
            if flag in step_content.get('used_flag', []):
                # Normalize the position by the total number of steps
                position = int(step_id) / len(steps)
                _tmp.append(position)
        if len(_tmp) > 0:
            avg_position = np.mean(_tmp)
            incorrect_positions.append(avg_position)
                
    # Calculate average positions
    if len(correct_positions) > 0:
        avg_correct_position = np.mean(correct_positions)
    else:
        avg_correct_position = -1
    if len(incorrect_positions) > 0:
        avg_incorrect_position = np.mean(incorrect_positions)
    else:
        avg_incorrect_position = -1
    
    # U test for significance
    u_stat, p_value = stats.mannwhitneyu(correct_positions, incorrect_positions, alternative='two-sided')
    return avg_correct_position, avg_incorrect_position, u_stat, p_value

data = get_all_records('./all/parsed/', use_human_anot='gemini-best', filter=lambda x: 'BEST' not in x or 'Common' in x)
print("Average Position Analysis for S.AR and S.HG:")
s_ar_avg_correct, s_ar_avg_incorrect, sig, p = average_position_analysis(data, 'Suggestion.Analogy_Recall')
print(f"S.AR - Correct: {s_ar_avg_correct}, Incorrect: {s_ar_avg_incorrect}, Significance: {sig}, p-value: {p}")
s_hg_avg_correct, s_hg_avg_incorrect, sig, p = average_position_analysis(data, 'Suggestion.Hypothesis_Generation')
print(f"S.HG - Correct: {s_hg_avg_correct}, Incorrect: {s_hg_avg_incorrect}, Significance: {sig}, p-value: {p}")


In [ ]:
data = get_all_records('machine_annotations/AIME/R1',use_human_anot='gemini-best')
analysis_once(data)

In [ ]:
data = get_all_records('machine_annotations/MATH/R1',use_human_anot='gemini-best')
analysis_once(data)

In [ ]:
data = get_all_records('human_annotations/R1',use_human_anot='gemini-best')

analysis_once(data)

In [ ]:
data = get_all_records('human_annotations/R1',use_human_anot=True)

analysis_once(data)

# Human annotation and used_flag consistency testing

In [ ]:
# calculate the f1 of each flag

#filter that no 'Peter' in the file name

machines = {
    'gemini': None,
    'gemini-best': None,
}
#machine_anots = get_all_records('./all/human_original/R1_reanot', use_human_anot=False)
for machine, _ in machines.items():
    machine_anots = get_all_records('./all/human_original/best_annots/R1', use_human_anot=machine)
    
    machines[machine] = machine_anots
    

human_anots = get_all_records('./all/human_original/best_annots/R1', use_human_anot=True)

# print number of all human_anot steps
steps = 0
for file, record in human_anots.items():
    for step_id, step_content in record['steps'].items():
        if 'used_flag' in step_content and step_content['used_flag'] is not None:
            steps += 1
            
print(f"Total number of human annotated steps: {steps}")


def calculate_f1_score(machine_anots, human_anots, flag_id):
    """
    Calculate F1 score for a specific flag across all records.
    
    Parameters:
    machine_anots -- Dictionary of machine annotations
    human_anots -- Dictionary of human annotations
    """

    true_positives = 0
    false_positives = 0
    false_negatives = 0
    numbers = 0
    for file in machine_anots.keys():
        machine_record = machine_anots[file]
        human_record = human_anots[file]
        
        for step_id, step_content in machine_record['steps'].items():
            machine_flag = step_content.get('used_flag', None)
            human_flag = human_record['steps'].get(step_id, {}).get('used_flag', None)
            if machine_flag is None or human_flag is None:
                continue
            if flags[flag_id] in machine_flag and flags[flag_id] in human_flag:
                numbers += 1
                true_positives += 1
            elif flags[flag_id] in machine_flag and flags[flag_id] not in human_flag:
                false_positives += 1
            elif flags[flag_id] not in machine_flag and flags[flag_id] in human_flag:
                false_negatives += 1
                numbers += 1
    # Calculate precision, recall, and F1 score
    if true_positives + false_positives == 0:
        precision = 0
    else:
        precision = true_positives / (true_positives + false_positives)
    if true_positives + false_negatives == 0:
        recall = 0
    else:
        recall = true_positives / (true_positives + false_negatives)
    if precision + recall == 0:
        f1_score = 0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)
    return precision, recall, f1_score, numbers, true_positives, false_positives, false_negatives
# Calculate F1 scores for each flag

results = {}
for flag_id in range(len(flags)):
    results[flags[flag_id]] = {}
    for machine, machine_anots in machines.items():
        precision, recall, f1_score, numbers, true_positives, false_positives, false_negatives = calculate_f1_score(machine_anots, human_anots, flag_id)
        results[flags[flag_id]][machine] = {
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score,
            'numbers': numbers,
            'true_positives': true_positives,
            'false_positives': false_positives,
            'false_negatives': false_negatives
        }
# Convert results to DataFrame for better visualization
f1_scores = []
for flag, scores in results.items():
    for machine, metrics in scores.items():
        f1_scores.append({
            'Flag': flag,
            'Machine': machine,
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1_Score': metrics['f1_score'],
            'Numbers': metrics['numbers'],
            'True_Positives': metrics['true_positives'],
            'False_Positives': metrics['false_positives'],
            'False_Negatives': metrics['false_negatives']
        })



In [ ]:
# Convert data to DataFrame
f1_df = pd.DataFrame(f1_scores)

# Extract major categories (Analysis, Inference, Judgment, Suggestion, Reflection)
f1_df['Category'] = f1_df['Flag'].apply(lambda x: x.split('.')[0])

# Calculate the weighted average F1 for each major category
# Use Numbers as weights
category_metrics = []

# First, calculate the frequency proportion of each major category
total_instances = f1_df['Numbers'].sum() / 3  # Divide by 3 because each sample is evaluated by 3 models
category_counts = f1_df.groupby('Category')['Numbers'].sum() / 3
category_percentages = category_counts / total_instances * 100

# Then calculate the weighted average F1 for each machine under each major category
for category in f1_df['Category'].unique():
    category_data = f1_df[f1_df['Category'] == category]
    
    # Calculate category proportion
    category_percentage = category_percentages[category]
    
    # Calculate the weighted average F1 for each machine in this category
    for machine in f1_df['Machine'].unique():
        machine_data = category_data[category_data['Machine'] == machine]
        if not machine_data.empty:
            # Calculate weighted average F1
            weighted_f1 = (machine_data['F1_Score'] * machine_data['Numbers']).sum() / machine_data['Numbers'].sum()
            
            category_metrics.append({
                'Category': category,
                'Machine': machine,
                'Weighted_F1': weighted_f1,
                'Category_Percentage': category_percentage
            })

category_metrics_df = pd.DataFrame(category_metrics)

# Add category percentage data
percentage_rows = []
for category in category_percentages.index:
    percentage_rows.append({
        'Category': category,
        'Machine': 'Category Percentage',
        'Weighted_F1': category_percentages[category],
        'Category_Percentage': category_percentages[category]
    })

# Use pd.concat instead of the deprecated append
category_metrics_df = pd.concat([category_metrics_df, pd.DataFrame(percentage_rows)], ignore_index=True)

# figures
os.makedirs('figures', exist_ok=True)

# Plot bar chart
plt.figure(figsize=(16, 9))

# Set the width of the groups and the width of the bars
n_groups = len(category_percentages)
bar_width = 0.2

# Set x-axis positions
index = np.arange(n_groups)

# Color settings
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

# Draw bars for each model
for i, (machine, color) in enumerate(zip(['Category Percentage', 'gemini', 'gemini-best'], colors)):
    data = category_metrics_df[category_metrics_df['Machine'] == machine]
    
    # Ensure data is arranged in the same order of categories
    data = data.set_index('Category').reindex(category_percentages.index).reset_index()
    
    if machine == 'Category Percentage':
        values = data['Category_Percentage']
        label = 'Category Percentage (%)'
    else:
        values = data['Weighted_F1'] * 100  # Convert to percentage form, consistent with category proportion
        label = f'{machine} F1 (%)'
    
    plt.bar(index + i*bar_width - 1.5*bar_width, values, bar_width, label=label, color=color, alpha=0.8)

# Add chart labels
plt.xlabel('Category', fontsize=14)
plt.ylabel('Percentage (%)', fontsize=14)
plt.xticks(index - 0.5*bar_width, category_percentages.index, rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Adjust chart margins
plt.tight_layout()

plt.savefig('figures/weighted_average_f1_scores_by_category.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Create a clearer data table
print("\nWeighted Average F1 Scores and Category Proportion by Category:")
# Rename Machine column for better readability
renamed_df = category_metrics_df.copy()
renamed_df.loc[renamed_df['Machine'] == 'Category Percentage', 'Machine'] = 'Category Percentage (%)'
renamed_df.loc[renamed_df['Machine'] != 'Category Percentage (%)', 'Weighted_F1'] *= 100  # Convert to percentage

# Create pivot table
pivot_table = renamed_df.pivot_table(
    index='Category', 
    columns='Machine', 
    values='Weighted_F1',
    aggfunc='first'
)

# Sort by category proportion in descending order
pivot_table = pivot_table.sort_values(by='Category Percentage (%)', ascending=False)

# Output table rounded to two decimal places
display(pivot_table.round(2))

# averaging f1 by weight
ave_f1 = {}
for i,machine in enumerate(pivot_table.columns[1:]):  # Skip the first column (Category Percentage)
    res = pivot_table.iloc[:, i+1].values * pivot_table.iloc[:,0].values / 100/100  # Convert to percentage
    ave_f1[machine] = res.sum()
print("\nAverage F1 Scores by Machine:")
ave_f1_df = pd.DataFrame(list(ave_f1.items()), columns=['Machine', 'Average F1 (%)'])
ave_f1_df['Average F1 (%)'] = ave_f1_df['Average F1 (%)'] * 100  # Convert to percentage
print(ave_f1_df.round(2))

In [ ]:
data = get_all_records('./all/parsed', use_human_anot='gemini-best', filter=lambda x: 'BEST' in x)

correct_record, incorrect_record = partition_samples(data)

# ，visualize2
colors = [
    "#FF5733", # Analysis.Problem_Definition
    "#FF6F33", # Analysis.Information_Organization
    "#FF8F33", # Analysis.Problem_Structuring
    "#33FF57", # Inference.Deductive_Reasoning
    "#33FF9F", # Inference.Inductive_Reasoning
    "#33FFCF", # Inference.Abductive_Reasoning
    "#333FFF", # Judgment.Principle_Selection
    "#337FFF", # Judgment.Evaluation_of_Alternatives,
    "#33BFFF", # Judgment.Conclusion_Decision
    "#AF33F1", # Suggestion.Strategic_Planning
    "#AF33BF", # Suggestion.Branch_Changing
    "#AF337F", # Suggestion.Hypothesis_Generation,
    "#AF3333", # Suggestion.Analogy_Recall
    "#E2F205", # Reflection.Self_Monitoring_Evaluation,
    "#C3D402", # Reflection.Counterfactual_Thinking
    "#98A501", # Reflection.Causal_Attribution
    "#6B8E03", # Reflection.Strategy_Regulation
    "#000000", # Not answered
]

def process_records_for_plot(records):
    """"""
    # 0-100
    normalized_data = []
    
    for file, record in records.items():
        steps_data = {}
        total_steps = len(record['steps'])
        
        if total_steps == 0:
            continue
            
        for step_id, step_content in record['steps'].items():
            step_number = int(step_id)
            normalized_step = int(step_number * 100 / total_steps)
            
            selected_flags = step_content.get('used_flag', [])
            
            # 
            if isinstance(selected_flags, str):
                try:
                    selected_flags = eval(selected_flags)
                except:
                    selected_flags = []
            
            # 
            flag_vector = np.zeros(len(flags))
            for flag_index, flag_name in enumerate(flags):
                if flag_name in selected_flags:
                    flag_vector[flag_index] = 1
                    
            # ，
            if flag_vector.sum() == 0:
                continue
                
            # 
            flag_vector = flag_vector / flag_vector.sum()
            
            steps_data[normalized_step] = flag_vector
        
        if steps_data:  # 
            file_data = np.ones((100, len(flags))) * -1
            last_step = 0
            
            # 
            for step, flag_vector in sorted(steps_data.items()):
                now_step = step
                file_data[last_step:now_step] = flag_vector
                last_step = now_step
                
            # 
            if last_step < 100:
                file_data[last_step:] = file_data[last_step-1] if last_step > 0 else np.zeros(len(flags))
                
            normalized_data.append(file_data)
    
    if not normalized_data:
        return None
        
    # 
    normalized_data = np.array(normalized_data)
    return normalized_data.mean(axis=0)

def plot_thinking_progress(data, title, filename):
    """"""
    plt.figure(figsize=(16, 8))
    
    # 
    sums = np.zeros(100)
    for i in reversed(range(len(flags))):
        # 
        label = flag_abbreviations[flags[i]]
        plt.fill_between(
            np.arange(100), 
            sums, 
            sums + data[:, i], 
            label=label, 
            color=colors[i] if i < len(colors) else 'gray'
        )
        sums += data[:, i]
    
    # 
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.gca().legend(
        list(reversed(handles)), 
        list(reversed(labels)), 
        loc='center right', 
        bbox_to_anchor=(1.25, 0.5), 
        fontsize=10
    )
    
    plt.xlabel('Progress')
    plt.ylabel('Portion')
    plt.xticks(np.arange(0, 101, 10))
    plt.xlim(0, 99)
    plt.ylim(0, 1.00)
    plt.grid(axis='y')
    
    # figuresPDF
    os.makedirs('figures', exist_ok=True)
    plt.savefig(f'figures/{filename}.pdf', bbox_inches='tight', dpi=600)
    plt.show()

# 
correct_data = process_records_for_plot(correct_record)
incorrect_data = process_records_for_plot(incorrect_record)

if correct_data is not None:
    plot_thinking_progress(correct_data, 'Correct Solutions Thinking Process', 'correct_thinking_progress')

if incorrect_data is not None:
    plot_thinking_progress(incorrect_data, 'Incorrect Solutions Thinking Process', 'incorrect_thinking_progress')